In [2]:
import os
import sys
import json
import torch
import pickle
import warnings
import numpy as np
import sympy as sp
import pandas as pd
import proplot as pplt
from IPython.display import display,HTML
sys.path.insert(0,'..')
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
MODELSDIR = CONFIGS['filepaths']['models']
SRCONFIG  = CONFIGS['experiments']['sr']
SEEDS     = SRCONFIG['seeds']

In [4]:
VARDICT = {
    'rh':r'\widehat{\mathrm{RH}}',
    'thetae':r'\widehat{\theta}_{e}',
    'thetaestar':r'{\widehat{\theta}_{e}^{*}}',
    'bl':r'\widetilde{\mathrm{B_L}}',
    'lf':r'\widetilde{\mathrm{LF}}',
    'shf':r'\widetilde{\mathrm{SHF}}',
    'lhf':r'\widetilde{\mathrm{LHF}}'}
SYMBOLS  = {k:sp.Symbol(k) for k in VARDICT}
FUNCDICT = {
    'min':sp.Min,
    'max':sp.Max,
    'abs':sp.Abs,
    'neg':lambda x:-x,
    'square':lambda x:x**2,
    'cube':lambda x:x**3,
    'exp':sp.exp}

TERMORDER = {'bl':0,'rh':1,'thetae':2,'thetaestar':3,'lf':4,'shf':5,'lhf':6}

def to_sympy(eq):
    return sp.sympify(eq,locals={**SYMBOLS,**FUNCDICT})

def round_nums(expr,ndigits=4):
    return expr.xreplace({n:sp.Float(round(float(n),ndigits),ndigits) for n in expr.atoms(sp.Float)})

def term_key(term):
    syms = term.free_symbols
    if not syms:return (99,str(term))
    names = sorted(s.name for s in syms)
    return (min(TERMORDER.get(n,50) for n in names),str(term))

def order_expr(expr):
    if expr.args:
        expr = expr.func(*[order_expr(arg) for arg in expr.args],evaluate=False)
    if isinstance(expr,sp.Add):
        expr = sp.Add(*sorted(sp.Add.make_args(expr),key=term_key),evaluate=False)
    return expr

def to_latex(expr):
    symnames = {SYMBOLS[k]:v for k,v in VARDICT.items()}
    latex = sp.latex(expr,symbol_names=symnames,mul_symbol='dot')
    latex = latex.replace(r'\left','').replace(r'\right','')
    return ' '.join(latex.split())

def prettify(eq):
    try:
        expr = to_sympy(str(eq).strip())
        expr = round_nums(expr)
        expr = order_expr(expr)
        return '$'+to_latex(expr)+'$'
    except Exception:
        return str(eq).strip()

def load_equations(runname):
    seedframes = {}
    for seed in SEEDS:
        filepath = os.path.join(MODELSDIR,'sr',f'{runname}_{seed}_equations.csv')
        if not os.path.exists(filepath):continue
        df = pd.read_csv(filepath)
        df['seed'] = seed
        seedframes[seed] = df
    return seedframes

def load_registry():
    filepath = os.path.join(MODELSDIR,'sr','optimized_equations.pkl')
    if not os.path.exists(filepath):return {}
    with open(filepath,'rb') as f:return pickle.load(f)

def prettify_optimized(opt):
    ns = {**SYMBOLS,**FUNCDICT}
    ns.update({k:sp.Float(v) for k,v in opt['constants'].items()})
    try:
        expr = eval(opt['form'],{'__builtins__':{}},ns)
        expr = round_nums(expr)
        expr = order_expr(expr)
        return r'$'+to_latex(expr)+'$'
    except Exception as e:
        return f"{opt['form']}  [render error: {e}]"

In [5]:
registry = load_registry()
rows = []
for name,eqspec in SRCONFIG['optimizedeqs'].items():
    opt = registry.get(name)
    if opt is None:
        rows.append({'Model':eqspec['description'],'Train MSE':'—','Valid MSE':'—'})
    else:
        rows.append({'Model':eqspec['description'],'Equation':prettify_optimized(opt),'Train MSE':f"{opt['train_loss']:.4f}",'Valid MSE':f"{opt['valid_loss']:.4f}"})
display(HTML(pd.DataFrame(rows).to_html(escape=False,index=False)))

Model,Equation,Train MSE,Valid MSE
SR-BL,$(\widetilde{\mathrm{B_L}} + 0.2957)^{3} + 0.1421$,0.4984,0.5017
SR-LO,$1.0 \cdot e^{\widehat{\mathrm{RH}}} - 0.6507$,0.6246,0.6214
SR-MED,"$1.558 \cdot \max(\widehat{\mathrm{RH}}, \widehat{\theta}_{e} - 1.471 \cdot {\widehat{\theta}_{e}^{*}} - 0.3756)^{3}$",0.4452,0.4522
SR-HI,"$(- 0.1271 \cdot \max(\widetilde{\mathrm{LF}}, \widetilde{\mathrm{SHF}}) + 1.307 \cdot \max(\widehat{\mathrm{RH}}, \widehat{\theta}_{e} - 1.359 \cdot {\widehat{\theta}_{e}^{*}} - 0.4173))^{3}$",0.4098,0.4188


In [ ]:
nn_cfg    = CONFIGS['experiments']['nn']
fieldvars = nn_cfg['runs']['nn_gauss']['fieldvars']
modeldir  = os.path.join(MODELSDIR,'nn')
seeds     = nn_cfg['seeds']

kernel_rows = []
for seed in seeds:
    ckpt_path = os.path.join(modeldir,f'nn_gauss_{seed}.pth')
    if not os.path.exists(ckpt_path):continue
    state = torch.load(ckpt_path,map_location='cpu')
    if 'state_dict' in state:state = state['state_dict']
    mu     = state['kernel.function.mu'].numpy()
    logstd = state['kernel.function.logstd'].numpy()
    for i,fv in enumerate(fieldvars):
        kernel_rows.append({
            'Variable':fv,
            'center':  round(float(0.75+mu[i]*0.25),4),
            'width':   round(float(np.exp(logstd[i])*0.25),4)})

if not kernel_rows:
    print('[warn] No nn_gauss checkpoints found — run on the cluster')
else:
    kdf  = pd.DataFrame(kernel_rows)
    means = kdf.groupby('Variable')[['center','width']].mean().round(4)
    means.index = ['$'+VARDICT[v]+'$' for v in means.index]
    means.columns = [r'$\sigma$ center',r'$\Delta\sigma$ width']
    display(HTML(means.to_html()))